In [1]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import timedelta
import os

In [2]:
df_deslizamientos = pd.read_csv('../data/datos_simma.csv')

In [3]:
#Separamos los casos de deslizamientos en cuatro grupos:
n = len(df_deslizamientos)
indices = np.linspace(0, n, 5, dtype=int)

df_deslizamientos1 = df_deslizamientos.iloc[indices[0]:indices[1]]
df_deslizamientos2 = df_deslizamientos.iloc[indices[1]:indices[2]]
df_deslizamientos3 = df_deslizamientos.iloc[indices[2]:indices[3]]
df_deslizamientos4 = df_deslizamientos.iloc[indices[3]:indices[4]]


In [4]:
print(len(df_deslizamientos1), len(df_deslizamientos2), len(df_deslizamientos3), len(df_deslizamientos4))

1407 1408 1408 1408


In [5]:
#pasamos las Fechas a formato datetime
df_deslizamientos['Fecha'] = pd.to_datetime(df_deslizamientos['Fecha'],format="%d/%m/%Y")
df_deslizamientos1['Fecha'] = pd.to_datetime(df_deslizamientos1['Fecha'],format="%d/%m/%Y")
df_deslizamientos2['Fecha'] = pd.to_datetime(df_deslizamientos2['Fecha'],format="%d/%m/%Y")
df_deslizamientos3['Fecha'] = pd.to_datetime(df_deslizamientos3['Fecha'],format="%d/%m/%Y")
df_deslizamientos4['Fecha'] = pd.to_datetime(df_deslizamientos4['Fecha'],format="%d/%m/%Y")

C:\Users\Alexander Sanguino\AppData\Local\Temp\ipykernel_9440\3225369252.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_deslizamientos1['Fecha'] = pd.to_datetime(df_deslizamientos1['Fecha'],format="%d/%m/%Y")
C:\Users\Alexander Sanguino\AppData\Local\Temp\ipykernel_9440\3225369252.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_deslizamientos2['Fecha'] = pd.to_datetime(df_deslizamientos2['Fecha'],format="%d/%m/%Y")
C:\Users\Alexander Sanguino\AppData\Local\Temp\ipykernel_9440\3225369252.py:5:

In [6]:
# Vamos a obtener los datos climáticos de la API de Open-Meteo para cada uno de los casos de deslizamientos.
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../scripts")))
from r import obtener_clima_30d

In [7]:
RUTA_SALIDA = '../data/processed/datos_con_clima_negativos.csv'

In [20]:
df_negativos = df_negativos[df_negativos["id_neg"].isin(ids_no_procesados)]
df_negativos.head()

,id_neg,Latitud,Longitud,Fecha
15605,94_month_2008,5.495556,-75.713056,2008-04-25
16037,3062_month_2004,6.256667,-72.304167,2004-07-07


In [22]:
df_negativos['Fecha'] = pd.to_datetime(df_negativos['Fecha'],format="%Y-%m-%d")

In [23]:
ids_procesados = set()
#if os.path.exists(RUTA_SALIDA):
 #   existentes = pd.read_csv(RUTA_SALIDA, usecols=["id"])
  #  ids_procesados = set(existentes["id"].unique())
   # print(f"👉 Ya existen {len(ids_procesados)} ids procesados previamente. Se omitirán.")

all_rows = []
count = 0
for _, row in df_negativos.iterrows():
    if row["id_neg"] in ids_procesados:
        continue

    lat, lon = row["Latitud"], row["Longitud"]
    fecha_evento = row["Fecha"]

    daily_data = obtener_clima_30d(lat, lon, fecha_evento)

    if daily_data:
        times = daily_data.get("time", [])

        for i in range(len(times)):
            all_rows.append({
                "id_neg": row["id_neg"],
                "fecha_evento": fecha_evento.strftime("%Y-%m-%d"),
                "fecha_clima": times[i],
                "temperature_2m_max": daily_data.get("temperature_2m_max", [None])[i],
                "temperature_2m_min": daily_data.get("temperature_2m_min", [None])[i],
                "temperature_2m_mean": daily_data.get("temperature_2m_mean", [None])[i],
                "precipitation_sum": daily_data.get("precipitation_sum", [None])[i],
                "relative_humidity_2m_max": daily_data.get("relative_humidity_2m_max", [None])[i],
                "relative_humidity_2m_min": daily_data.get("relative_humidity_2m_min", [None])[i],
                "relative_humidity_2m_mean": daily_data.get("relative_humidity_2m_mean", [None])[i],
                "pressure_msl_mean": daily_data.get("pressure_msl_mean", [None])[i],
                "wind_speed_10m_max": daily_data.get("wind_speed_10m_max", [None])[i],
                "et0_fao_evapotranspiration": daily_data.get("et0_fao_evapotranspiration", [None])[i]
            })

    count += 1
    if count % 100 == 0:
        # Guardado parcial para no perder progreso
        modo = "a" if os.path.exists(RUTA_SALIDA) else "w"
        header = not os.path.exists(RUTA_SALIDA)
        pd.DataFrame(all_rows).to_csv(RUTA_SALIDA, mode=modo, index=False, header=header)
        all_rows = []
        print(f"Guardado parcial para {count} registros procesados.")
        time.sleep(60)  # Espera para no saturar la API

In [24]:
if all_rows:
    modo = "a" if os.path.exists(RUTA_SALIDA) else "w"
    header = not os.path.exists(RUTA_SALIDA)
    pd.DataFrame(all_rows).to_csv(RUTA_SALIDA, mode=modo, index=False, header=header)

print(f"Finalizado. Resultado guardado en {RUTA_SALIDA}")

Finalizado. Resultado guardado en ../data/processed/datos_con_clima_negativos.csv


In [25]:
df= pd.read_csv(RUTA_SALIDA)
df_negativos = pd.read_csv('../data/processed/datos_negativos.csv')

In [26]:
ids_esperados = df_negativos["id_neg"].unique()
ids_no_procesados = set(ids_esperados) - set(df["id_neg"].unique())
print(f"IDs no procesados en absoluto: {ids_no_procesados}")

IDs no procesados en absoluto: set()


In [27]:
len(ids_no_procesados)

0

In [87]:
#Hay que crear los casos negativos, que son los casos de deslizamientos que no ocurrieron.
from r import generar_eventos_negativos

In [170]:
anos_totales=range(2000, 2024)
df_negativos = generar_eventos_negativos(df_deslizamientos,anos_totales)

HOLAAA


In [14]:
len(df_negativos)

2368

In [172]:
#establecemos una relación 1:3 entre casos positivos y negativos
df_negativos = df_negativos.sample(n=16800, random_state=42)

In [177]:
df_negativos.to_csv('../data/processed/datos_negativos.csv', index=False)